# 00. Importación y limpieza

Este notebook llama a `src/`, comprueba las cifras de control de `references/plan_datos.md` y escribe las cuatro tablas de `data/processed/`.

ACODECO y el MEF quedan en la misma tabla de costo, con `fuente` distinta. No se promedian. Las dos bases del IPC tampoco se empalman.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / "src").is_dir():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

from esquemas import validar
from io_acodeco import leer_costo_historico, leer_precios
from io_inec import leer_ipc
from io_mef import leer_costo_mef
from io_salarios import leer_salarios

PROCESSED = RAIZ / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## Precios por supermercado

CSV de 2019 (marzo - diciembre) y 2022 (enero - junio), más la segunda quincena de septiembre de 2026. Cada mes de CSV debe traer 59 productos.

In [2]:
precios = leer_precios()
csv = precios.loc[precios["fuente"].eq("acodeco_csv")].copy()
pdf = precios.loc[precios["fuente"].eq("acodeco_pdf_sep2026")].copy()

productos_por_mes = csv.groupby(["anio", "mes"])["producto_estandar"].nunique()
if not (productos_por_mes == 59).all():
    raise AssertionError(f"Un mes de CSV no tiene 59 productos:\n{productos_por_mes[productos_por_mes != 59]}")
if (csv["precio"] == 0).any() or (pdf["precio"] == 0).any():
    raise AssertionError("Hay un precio en 0. Los faltantes deben quedar en NaN.")
if csv["precio"].isna().sum() == 0:
    raise AssertionError("Los CSV no dejaron ningún precio vacío; se esperaban faltantes en NaN.")
meses_csv = {(int(anio), int(mes)) for anio, mes in productos_por_mes.index}
esperado_csv = {(2019, mes) for mes in range(3, 13)} | {(2022, mes) for mes in range(1, 7)}
if meses_csv != esperado_csv:
    raise AssertionError(f"Meses de CSV inesperados: {sorted(meses_csv)}")
if pdf["quincena"].nunique() != 1 or int(pdf["quincena"].iloc[0]) != 2:
    raise AssertionError("El PDF de septiembre 2026 no quedó como segunda quincena.")
if pdf["producto_estandar"].nunique() != 59:
    raise AssertionError("El PDF de septiembre 2026 no tiene 59 productos.")

print(f"CSV: {len(csv):,} filas, {len(meses_csv)} meses, {int(csv['precio'].isna().sum()):,} precios vacíos")
print(f"PDF septiembre 2026: {len(pdf):,} filas, {pdf['establecimiento_estandar'].nunique()} establecimientos")
productos_por_mes.rename("productos").to_frame()

CSV: 84,134 filas, 16 meses, 27,076 precios vacíos
PDF septiembre 2026: 5,015 filas, 82 establecimientos


productos
anio mes           
2019 3           59
     4           59
     5           59
     6           59
     7           59
     8           59
     9           59
     10          59
     11          59
     12          59
2022 1           59
     2           59
     3           59
     4           59
     5           59
     6           59

## Costo mensual de la canasta

El histórico de ACODECO cubre todo 2019, 2022 y 6 meses de 2026 (enero - agosto). El MEF cubre los mismos años, sin agosto de 2022, y enero y febrero de 2026 salen como dos filas. Las dos series se concatenan y se distinguen por `fuente`.

In [3]:
def costo_de(tabla, anio, mes):
    fila = tabla.loc[tabla["anio"].eq(anio) & tabla["mes"].eq(mes), "costo_total"]
    if len(fila) != 1:
        raise AssertionError(f"Se esperaba un costo para {anio}-{mes:02d} y hay {len(fila)}")
    return float(fila.iloc[0])


def coincide(tabla, anio, mes, esperado):
    valor = costo_de(tabla, anio, mes)
    if abs(valor - esperado) > 0.001:
        raise AssertionError(f"{anio}-{mes:02d}: {valor} != {esperado}")


acodeco = leer_costo_historico()
mef = leer_costo_mef()

if acodeco.groupby("anio").size().to_dict() != {2019: 12, 2022: 12, 2026: 8}:
    raise AssertionError(f"Meses ACODECO: {acodeco.groupby('anio').size().to_dict()}")
for anio, mes, esperado in [
    (2019, 1, 262.93), (2019, 6, 260.64), (2019, 8, 263.09), (2019, 12, 263.52),
    (2022, 1, 278.87), (2022, 6, 285.92), (2022, 8, 285.06), (2022, 12, 287.41),
    (2026, 1, 304.40), (2026, 6, 303.45), (2026, 8, 304.85),
]:
    coincide(acodeco, anio, mes, esperado)
if ((acodeco["anio"] == 2026) & (acodeco["mes"] > 8)).any():
    raise AssertionError("El histórico de 2026 pasa de agosto")

if mef.groupby("anio").size().to_dict() != {2019: 12, 2022: 11, 2026: 8}:
    raise AssertionError(f"Meses MEF: {mef.groupby('anio').size().to_dict()}")
if ((mef["anio"] == 2022) & (mef["mes"] == 8)).any():
    raise AssertionError("Agosto de 2022 no está publicado y no debe aparecer")
if set(mef.loc[mef["anio"].eq(2026) & mef["mes"].isin([1, 2]), "mes"]) != {1, 2}:
    raise AssertionError("Enero y febrero de 2026 deben ser dos filas")
for anio, mes, esperado in [
    (2019, 1, 306.08), (2019, 6, 308.99), (2019, 12, 306.68),
    (2022, 1, 318.12), (2022, 6, 330.40), (2026, 8, 369.21),
]:
    coincide(mef, anio, mes, esperado)

costo = pd.concat([acodeco, mef], ignore_index=True)
costo = costo.sort_values(["fuente", "fecha"]).reset_index(drop=True)
costo = validar(costo, "costo_canasta_mensual")
if costo.duplicated(["fuente", "anio", "mes"]).any():
    raise AssertionError("Hay dos costos de la misma fuente para el mismo mes")
ambos = costo.groupby(["anio", "mes"])["fuente"].nunique()
if (ambos == 2).sum() == 0:
    raise AssertionError("Las dos fuentes no quedaron en paralelo")

print(costo.groupby(["fuente", "anio"]).size().unstack(fill_value=0))
costo.loc[costo["mes"].isin([1, 6, 8, 12]), ["fecha", "fuente", "costo_total"]]

anio               2019  2022  2026
fuente                             
acodeco_historico    12    12     8
mef_cbfa             12    11     8


,fecha,fuente,costo_total
0,2019-01-01,acodeco_historico,262.93
5,2019-06-01,acodeco_historico,260.64
7,2019-08-01,acodeco_historico,263.09
11,2019-12-01,acodeco_historico,263.52
12,2022-01-01,acodeco_historico,278.87
17,2022-06-01,acodeco_historico,285.92
19,2022-08-01,acodeco_historico,285.06
23,2022-12-01,acodeco_historico,287.41
24,2026-01-01,acodeco_historico,304.40
29,2026-06-01,acodeco_historico,303.45


## IPC

2019 y 2022 vienen del Cuadro 6, base 2013 = 100, distritos de Panamá y San Miguelito. Diciembre 2025octubre 2026 vienen del Cuadro 8, base 2024 = 100, provincia de Panamá. Cada año usa una sola base. Este notebook no calcula una variación que cruce las dos.

In [4]:
ipc = leer_ipc()
if ipc["base"].isna().any() or ipc["base"].eq("").any():
    raise AssertionError("Hay filas de IPC sin base")
bases_por_anio = ipc.groupby("anio")["base"].nunique()
if (bases_por_anio != 1).any():
    raise AssertionError(f"Un año mezcla bases:\n{bases_por_anio}")
if set(ipc.loc[ipc["anio"].isin([2019, 2022]), "base"]) != {"2013=100"}:
    raise AssertionError("2019 y 2022 deben estar en base 2013=100")
if set(ipc.loc[ipc["anio"].eq(2026), "base"]) != {"2024=100"}:
    raise AssertionError("2026 debe estar en base 2024=100")

def indice(anio, mes, grupo):
    fila = ipc.loc[ipc["anio"].eq(anio) & ipc["mes"].eq(mes) & ipc["grupo"].eq(grupo), "indice"]
    if len(fila) != 1:
        raise AssertionError(f"IPC {anio}-{mes} {grupo}: {len(fila)} filas")
    return round(float(fila.iloc[0]), 1)

if indice(2019, 1, "total") != 103.8 or indice(2022, 6, "alimentos_y_bebidas_no_alcoholicas") != 106.3:
    raise AssertionError("La muestra del Cuadro 6 no coincide")
if indice(2026, 10, "total") != 102.9:
    raise AssertionError("Octubre 2026 total no coincide con el Cuadro 8")
nota_octubre = ipc.loc[ipc["anio"].eq(2026) & ipc["mes"].eq(10), "notas"]
if not nota_octubre.str.contains("3.3").all() or not nota_octubre.str.contains("2.6").all():
    raise AssertionError("Falta la variación interanual de octubre 2026 en notas")

print(ipc.groupby(["anio", "base", "geografia"]).size())
ipc.loc[ipc["mes"].isin([1, 6, 10, 12]), ["fecha", "grupo", "indice", "base", "geografia"]]

anio  base      geografia             
2019  2013=100  panama_y_san_miguelito    24
2022  2013=100  panama_y_san_miguelito    24
2025  2024=100  provincia_de_panama        2
2026  2024=100  provincia_de_panama       20
dtype: int64


,fecha,grupo,indice,base,geografia
0,2019-01-01,alimentos_y_bebidas_no_alcoholicas,102.100000,2013=100,panama_y_san_miguelito
1,2019-01-01,total,103.800000,2013=100,panama_y_san_miguelito
10,2019-06-01,alimentos_y_bebidas_no_alcoholicas,101.800000,2013=100,panama_y_san_miguelito
11,2019-06-01,total,104.300000,2013=100,panama_y_san_miguelito
18,2019-10-01,alimentos_y_bebidas_no_alcoholicas,102.100000,2013=100,panama_y_san_miguelito
19,2019-10-01,total,104.200000,2013=100,panama_y_san_miguelito
22,2019-12-01,alimentos_y_bebidas_no_alcoholicas,101.600000,2013=100,panama_y_san_miguelito
23,2019-12-01,total,104.000000,2013=100,panama_y_san_miguelito
24,2022-01-01,alimentos_y_bebidas_no_alcoholicas,103.600000,2013=100,panama_y_san_miguelito
25,2022-01-01,total,105.100000,2013=100,panama_y_san_miguelito


## Salario mínimo

Tasas ya verificadas de los tres decretos. El mensual del comercio es hora × 208. El doméstico usa el monto oficial. En 2019 no existe la categoría de supermercados de 5 o más sucursales.

In [5]:
salario = leer_salarios()
if ((salario["anio_analisis"] == 2019) & (salario["categoria"] == "supermercados_5_plus")).any():
    raise AssertionError("2019 no debe traer supermercados de 5 o más sucursales")
r1 = salario.loc[
    (salario["anio_analisis"] == 2022)
    & (salario["region"] == "1")
    & (salario["categoria"] == "gran_empresa"),
    "salario_mensual",
]
if abs(float(r1.iloc[0]) - 584.48) > 0.001:
    raise AssertionError(f"2022 Región 1 gran empresa debía ser 584.48 y es {float(r1.iloc[0])}")
oficial = salario.loc[salario["es_oficial_mensual"], "actividad"].unique()
if list(oficial) != ["trabajo_domestico"]:
    raise AssertionError(f"El mensual oficial solo aplica al doméstico: {oficial}")
if salario.loc[salario["actividad"].eq("trabajo_domestico"), "salario_hora"].notna().any():
    raise AssertionError("El doméstico no trae tasa horaria")

print(f"{len(salario)} filas")
salario.loc[
    salario["actividad"].eq("comercio_al_por_menor"),
    ["anio_analisis", "region", "categoria", "salario_hora", "salario_mensual"],
]

22 filas


,anio_analisis,region,categoria,salario_hora,salario_mensual
0,2019,1,pequena,2.20,457.60
1,2019,1,gran_empresa,2.72,565.76
2,2019,2,pequena,1.84,382.72
3,2019,2,gran_empresa,2.23,463.84
4,2022,1,pequena,2.27,472.16
5,2022,1,gran_empresa,2.81,584.48
6,2022,1,supermercados_5_plus,2.88,599.04
7,2022,2,pequena,1.90,395.20
8,2022,2,gran_empresa,2.30,478.40
9,2022,2,supermercados_5_plus,2.36,490.88


## Tablas para el análisis

Los notebooks siguientes leen solo estos CSV.

In [6]:
def guardar(tabla, nombre):
    salida = tabla.copy()
    for columna in salida.columns:
        if pd.api.types.is_datetime64_any_dtype(salida[columna]):
            salida[columna] = salida[columna].dt.strftime("%Y-%m-%d")
    ruta = PROCESSED / nombre
    salida.to_csv(ruta, index=False)
    print(f"{nombre}: {len(salida):,} filas -> {ruta.relative_to(RAIZ)}")


guardar(ipc, "ipc.csv")
guardar(costo, "costo_canasta_mensual.csv")
guardar(salario, "salario_minimo.csv")
guardar(precios, "precios_supermercados.csv")

ipc.csv: 70 filas -> data/processed/ipc.csv
costo_canasta_mensual.csv: 63 filas -> data/processed/costo_canasta_mensual.csv
salario_minimo.csv: 22 filas -> data/processed/salario_minimo.csv
precios_supermercados.csv: 89,149 filas -> data/processed/precios_supermercados.csv
